In [36]:
import duckdb
import pandas as pd
from pathlib import Path
import gc
import yaml
import logging


pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', True)

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True 
)


In [37]:
class ConnectionManager():
    def __init__(self, db_con_str):
        # la fonction duckdb.connect transforme le path en absolue, autant le faire ici 
        self.db_con_str = Path(db_con_str).expanduser().resolve()

        # connexion lazy
        self._con = None


    @property
    def con(self):
        if(self._con is None):
            # se connecter à la base de données
            self._con = duckdb.connect(self.db_con_str)

        return self._con


    @con.setter
    def con(self, value):
        # si au moment de changer la connexion on a déjà une connexion active
        if(self._con is not None):
            self._con.close() # cloturer la connexion en cours
            self._con = None # retirer la référence sur la connexion en cours
            gc.collect() # appeler le garbage collector pour forcer l'action de libérer les ressources et éviter les conflits d'accès

        self._con = value # pointer sur la nouvelle connexion
    
    
    def close_con(self):
        # on exploite le setter de la propriété pour cloturer correctement la connexion
        self.con = None


    def __del__(self):
        """Ferme automatiquement la connexion DuckDB quand l'objet est détruit."""
        try:
            self.close_con()
        except Exception:
            pass


    def __enter__(self):
        return self
    

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close_con()
        return False

In [38]:
class ConnectionUtils(ConnectionManager):
    def __init__(self, db_con_str : str):
        super().__init__(db_con_str)


    def tables(self):
        """Retourne la liste de toutes les tables physiques de la base courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            AND table_type = 'BASE TABLE'
            ORDER BY table_name
        """)


    def views(self):
        """Cette fonction renvoi la liste de toutes vues accessibles dans la base de données courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.views
            WHERE table_catalog = current_database()
            ORDER BY table_name
        """)


    def tables_views(self):
        """Retourne la liste des tables et des vues de la base courante"""
        return self.con.sql("""
            SELECT 
                table_name,
                table_type          -- 'BASE TABLE' ou 'VIEW'
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            ORDER BY table_type, table_name
        """)


    def table_exists(self, table_name : str):
        """Cette foction vérife qu'une table physique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                (table_name = '{table_name}')
                AND
                (table_type = 'BASE TABLE')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]
    
    
    def view_exists(self, view_name : str):
        """Cette fonction check si une vue existe"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.views 
            WHERE 
                (table_name = '{view_name}')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]


    def table_view_exists(self, name : str):
        """Cette foction vérife qu'une table physique ou une vue logique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                (table_name = '{name}')
                AND
                (table_type = 'BASE TABLE' OR table_type = 'VIEW')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]


    def drop_table_if_exists(self, table_name : str):
        """Cette fonction permet de supprimer une table s'elle existe"""
        self.con.sql(f"DROP TABLE IF EXISTS {table_name}")


    def drop_tables_if_exists(self, tables : list[str]):
        """Cette fonction surpprime chaque table de la liste tables s'elle existe dans la base courante"""
        for table_name in tables : 
            self.drop_table_if_exists(table_name)


    def drop_view_if_exists(self, view_name : str):
        """Cette fonction permet de supprimer une vue s'elle existe"""
        self.con.sql(f"DROP VIEW IF EXISTS {view_name}")


    def drop_views_if_exists(self, views : list[str]):
        """Cette fonction surpprime chaque vue de la liste views s'elle existe dans la base courante"""
        for view_name in views : 
            self.drop_view_if_exists(view_name)


    def table(self, table_name : str):
        """Cette fonction renvoi la table dont le nom est passé en paramètre"""
        return self.con.table(table_name)


    def view(self, view_name : str):
        """Cette fonction renvoi la vue dont le nom est passé en paramètre"""
        return self.con.view(view_name)


    def table_view(self, name : str):
        """Retourne la relation d'une table ou d'une vue selon ce qui existe."""
        return self.con.sql(f"SELECT * FROM {name}")


    def create_table_view_if_not_exists(self, name: str, sql: str, type: str = "VIEW"):
        """
        Crée une vue ou une table uniquement si elle n'existe pas encore.
        """
        sql = sql.strip().rstrip(";")
        self.con.sql(f"""CREATE {type} IF NOT EXISTS {name} AS ({sql})""")
        

In [39]:
class ConnectionPipeline(ConnectionUtils):
    def __init__(self, db_con_str : str, pipeline_file_path : str):
        super().__init__(db_con_str)
        self.pipeline_file_path = Path(pipeline_file_path).expanduser().resolve()
        self._pipeline = None


    def load_pipeline(self) -> dict:
        """Charge le fichier de définition des tables/vues."""
        with open(self.pipeline_file_path, "r", encoding="utf-8") as f:
            pipeline = yaml.safe_load(f)
            return pipeline


    @property
    def pipeline(self):
        if(self._pipeline is None):
            self._pipeline = self.load_pipeline()
        return self._pipeline
    

    def df_from_file(self, file: str | Path, **kwargs) -> pd.DataFrame:
        """Charge un fichier en DataFrame selon son extension + options"""
        path = Path(file).expanduser().resolve()
        suffix = path.suffix.lower()

        if suffix in {".xlsx", ".xls", ".xlsm"}:
            return pd.read_excel(path, **kwargs)

        elif suffix == ".csv":
            return pd.read_csv(path, **kwargs)

        elif suffix == ".tsv":
            return pd.read_csv(path, sep="\t", **kwargs)

        elif suffix == ".json":
            return pd.read_json(path, **kwargs)

        elif suffix == ".parquet":
            return pd.read_parquet(path, **kwargs)

        else:
            raise ValueError(f"Extension non supportée : {suffix}")


    def df_from_file_config(self, config : dict):
        # On prépare les kwargs en enlevant les clés réservées
        reserved = {"type", "requires", "file"}
        kwargs = {k: v for k, v in config.items() if k not in reserved}

        return self.df_from_file(config["file"], **kwargs)


    def process_dataframe_type(self, name : str):
        if(name in self.pipeline):
            if(not self.table_view_exists(name)):
                config = self.pipeline[name]
                df = self.df_from_file_config(config)
                self.con.register(name, df)


    def process_table_view_type(self, name : str):
        if(name in self.pipeline):
            config = self.pipeline[name]
            self.create_table_view_if_not_exists(name, config["sql"], config["type"])


    def process(self, name : str):
        logging.getLogger().debug(f"process({name})")

        if(name in self.pipeline):
            config = self.pipeline[name]

            if(config["type"] == "dataframe"):
                self.process_dataframe_type(name)

            elif(config["type"] in ["table", "view"]):
                self.process_table_view_type(name)


    def process_with_requires(self, name : str):
        if(name in self.pipeline):
            config = self.pipeline[name]

            for subname in config.get("requires", []):
                self.process_with_requires(subname)

            self.process(name)


    def p_table_view(self, name : str):
        self.process_with_requires(name)
        return self.table_view(name)

In [ ]:
cp = ConnectionPipeline(db_con_str = "duckdb/pilotes/base/base.duckdb", pipeline_file_path = "config/pipeline.yaml")
cp.p_table_view("v_sales_model_base")

2026-08-04 03:30:33,350 | DEBUG | process(df_sales)


2026-08-04 03:31:20,055 | DEBUG | process(t_sales)
2026-08-04 03:31:20,243 | DEBUG | process(df_sales)


In [26]:
cp.tables_views()

┌───────────────────────────────┬────────────┐
│          table_name           │ table_type │
│            varchar            │  varchar   │
├───────────────────────────────┼────────────┤
│ t_sales                       │ BASE TABLE │
│ t_sales_model                 │ BASE TABLE │
│ t_sales_model_base            │ BASE TABLE │
│ v_gamme_paniers_ventes        │ VIEW       │
│ v_gamme_paniers_ventes_mois   │ VIEW       │
│ v_nombre_mois                 │ VIEW       │
│ v_paniers                     │ VIEW       │
│ v_paniers_ventes              │ VIEW       │
│ v_paniers_ventes_mois         │ VIEW       │
│ v_produit_paniers_ventes      │ VIEW       │
│ v_produit_paniers_ventes_mois │ VIEW       │
│ v_refs                        │ VIEW       │
│ v_sales                       │ VIEW       │
│ v_sales_model                 │ VIEW       │
│ v_sales_model_base            │ VIEW       │
│ v_type_paniers_ventes         │ VIEW       │
│ v_type_paniers_ventes_mois    │ VIEW       │
└────────────

In [ ]:


class SalesPilBase(ConnectionManager):
    def __init__(self, db_con_str : str = "duckdb/pilotes/base/base.duckdb", sales_file_path : str = "data/hotel_sales_raw_extended_data.xlsx"):
        super().__init__(db_con_str)

        # initialement la base de données est vide et le point de départ est un extract au format fichier des données
        self.sales_file_path = Path(sales_file_path).expanduser().resolve()

        
    
    


    
    

    def create_if_not_exists_view(self, view_name : str):
        """Cette fonction crée une vue s'elle n'existe pas"""
        if(not self.view_exists(view_name)):
            getattr(self, f"create_or_replace_view_{view_name}")()
    
    
    def create_if_not_exists_views(self, view_names : list[str]):
        """Cette fonction crée chaque vue de la liste view_names s'elle n'existe pas"""
        for view_name in view_names:
            self.create_if_not_exists_view(view_name)


    def view_df(self, view_name : str):
        """Cette fonction permet de récupérer une vue ou une table sous forme d'une dataframe pandas"""

        # s'assurer que la vue source existe
        self.create_if_not_exists_view(view_name)

        # récupérer les données de la vue sous forme de dataframe
        df = self.con.sql(f"SELECT * FROM {view_name}").df()

        # restituer la dataframe
        return df


    


    def create_or_replace_table_sales(self):
        """Cette fonction permet de créer ou de remplacer la table t_sales correspondant aux ventes brutes"""
        df = pd.read_excel(self.sales_file_path) # lire le fichier excel des ventes
        self.con.register("df", df) # enregistrer la dataframe dans le catalogue
        self.con.sql("CREATE OR REPLACE TABLE t_sales AS SELECT * FROM df") # créer une table physique dans la base de données à partir de la dataframe


    def create_if_not_exists_table_sales(self):
        """Cette fonction crée la table des ventes brutes s'elle n'existe pas"""
        if(not self.table_exists("t_sales")):
            self.create_or_replace_table_sales()



    def create_or_replace_view_v_paniers(self):
        """Cette fonction permet de calculer des indicateurs au niveau de granularité panier"""

        # s'assurer que la table des ventes existe
        self.create_if_not_exists_table_sales()

        # calculer les indicateurs par panier
        sim.con.sql("""
            CREATE OR REPLACE VIEW v_paniers AS (
                SELECT
                    HOTEL_CODE,
                    ORDER_ID,
                    SUM(QUANTITE) AS nombre_ventes_par_panier,
                    COUNT(DISTINCT NOM_PRODUIT) AS nombre_produits_par_panier,
                    COUNT(DISTINCT GAMME) AS nombre_gammes_par_panier,
                    COUNT(DISTINCT TYPE) AS nombre_types_par_panier,

                    SUM(PRIX_TTC) AS montant_ventes_par_panier,
                    SUM(PRIX_TTC_MARCHE) AS montant_achats_par_panier,
                    SUM(MARGE) AS montant_marge_par_panier
                FROM
                    t_sales
                GROUP BY
                    HOTEL_CODE,
                    ORDER_ID
            )
        """)

        
    def create_or_replace_view_v_sales(self):
        """
        Cette fonction crée la vue sur la table des ventes. 
        L'objectif est de mettre en place des flux qui s'appuient uniquement sur les vues.
        L'intérêt est que sans modification des données de tables phyisiques, il suffit juste de modifier les vues du flux pour obtenir des résultats différents.
        """

        # s'assurer que la table physique des ventes brutes existe
        self.create_if_not_exists_table_sales()

        # créer une vue sur la table physique
        self.con.sql("""
            CREATE OR REPLACE VIEW v_sales AS (
                SELECT 
                    * 
                FROM 
                    t_sales
                LEFT JOIN
                    v_paniers
                USING
                    (HOTEL_CODE, ORDER_ID)
            )
        """)


    def create_or_replace_table_from_view(self, table_name : str, view_name : str):
        """Cette fonction permet de créer ou de remplacer physiquement une table a partir d'une vue existante"""
        self.con.sql(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM {view_name}")


    def create_if_not_exists_table_from_view(self, table_name : str, view_name : str):
        """Cette fonction permet de créer physiquement une table s'elle n'existe pas a partir d'une vue existante"""
        self.con.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM {view_name}")


    def create_or_replace_view_v_sales_model(self):
        """Cette fonction permet de créer la vue sur les ventes qui vont être utilisées pour la modélisation"""

        # s'assurer que la vue sue les données brutes existe afin de pouvoir créer la vue sur les données de modélisation
        self.create_if_not_exists_view("v_sales")

        # données des ventes utilisées pour la modélisation
        self.con.sql("""
            CREATE OR REPLACE VIEW v_sales_model AS
            (
                SELECT 
                    * 
                FROM 
                    v_sales 
                WHERE 
                    (YEAR(DATE) < (SELECT YEAR(MAX(DATE)) FROM v_sales))
                    AND 
                    (STATUT = 'DONE')
                    AND 
                    (QUANTITE > 0)
                    AND
                    (TYPE IS NOT NULL)
                    AND
                    (GAMME IS NOT NULL)
                    AND
                    (NOM_PRODUIT IS NOT NULL)
                    
            );
        """)


    def create_or_replace_view_v_refs(self):
        """Cette fonction permet de créer une vue référentielle des hotels avec leurs infos identifiantes"""

        # s'assurer que la vue sur les données utilisées pour la modélisation existe
        self.create_if_not_exists_view("v_sales_model")

        # créer la vue référentielle à partir des données de modélisation
        self.con.sql("""
            CREATE OR REPLACE VIEW v_refs AS (
            SELECT DISTINCT
                HOTEL_CODE AS hotel_code,
                NOM_BOUTIQUE AS hotel_name,
                SOLUTION AS solution,
                METRES_LINEAIRES AS metres_lineaires
            FROM 
                v_sales_model
            ORDER BY 
                hotel_code,
                hotel_name,
                solution,
                metres_lineaires
        );
        """)


    def create_or_replace_view_v_nombre_mois(self):
        """Cette fonction permet de créer une vue permettant d'obtenir le nombre de mois d'activités pour chaque hotel"""

        # s'assurer que la vue sur les données utilisées pour la modélisation existe
        self.create_if_not_exists_view("v_sales_model")

        # nombre de mois d'activités de ventes par hotel
        self.con.sql("""
            CREATE OR REPLACE VIEW v_nombre_mois AS (
            SELECT
                HOTEL_CODE AS hotel_code,
                DATE_DIFF('month', MIN(DATE), MAX(DATE)) + 1 AS nombre_mois
            FROM 
                v_sales_model
            GROUP BY
                HOTEL_CODE
            ORDER BY 
                hotel_code
        );
        """)


    def create_or_replace_view_v_produits(self):
        """Cette fonction crée une vue sur la liste des produits"""

        # s'assurer que la vue sur les données utilisées pour la modélisation existe
        self.create_if_not_exists_view("v_sales_model")

        # référentiel produit par hotels
        self.con.sql("""
            CREATE OR REPLACE VIEW v_produits AS (
            SELECT DISTINCT
                HOTEL_CODE AS hotel_code,
                NOM_PRODUIT AS produit
            FROM 
                v_sales_model
            ORDER BY 
                hotel_code,
                produit
        );
        """)


    def create_or_replace_view_v_gammes(self):
        """Cette fonction crée une vue sur la liste des gammes"""

        # s'assurer que la vue sur les données utilisées pour la modélisation existe
        self.create_if_not_exists_view("v_sales_model")

        # référentiel produit par hotels
        self.con.sql("""
            CREATE OR REPLACE VIEW v_gammes AS (
            SELECT DISTINCT
                HOTEL_CODE AS hotel_code,
                GAMME AS gamme
            FROM 
                v_sales_model
            ORDER BY 
                hotel_code,
                gamme
        );
        """)


    def create_or_replace_view_v_types(self):
        """Cette fonction crée une vue sur la liste des types"""

        # s'assurer que la vue sur les données utilisées pour la modélisation existe
        self.create_if_not_exists_view("v_sales_model")

        # référentiel produit par hotels
        self.con.sql("""
            CREATE OR REPLACE VIEW v_types AS (
            SELECT DISTINCT
                HOTEL_CODE AS hotel_code,
                TYPE AS type
            FROM 
                v_sales_model
            ORDER BY 
                hotel_code,
                type
        );
        """)


    



    def create_or_replace_view_v_produit_paniers_ventes(self): 
        """Cette fonction permet de calculer des indicateurs au niveau de granularité produit"""

        # s'assurer que la vue sur les données de modélisation existe
        self.create_if_not_exists_views(["v_sales_model"])
                    
        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme et produit
        self.con.sql("""
            CREATE OR REPLACE VIEW v_produit_paniers_ventes AS (
            SELECT 
                HOTEL_CODE AS hotel_code,
                TYPE as type,
                GAMME AS gamme,
                NOM_PRODUIT AS produit,

                SUM(QUANTITE) AS produit_nombre_ventes,
                COUNT(DISTINCT ORDER_ID) AS produit_nombre_paniers,

                SUM(PRIX_TTC) AS produit_montant_ventes,
                SUM(PRIX_TTC_MARCHE) AS produit_montant_achats,
                SUM(MARGE) AS produit_montant_marge,

                SUM(PRIX_TTC) / SUM(QUANTITE) AS produit_montant_par_vente,
                SUM(PRIX_TTC_MARCHE) / SUM(QUANTITE) AS produit_montant_achats_par_vente,
                SUM(MARGE) / SUM(QUANTITE) AS produit_montant_marge_par_vente,

                
                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS produit_nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS produit_montant_ventes_par_panier,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT ORDER_ID) AS produit_montant_achats_par_panier,
                SUM(MARGE) / COUNT(DISTINCT ORDER_ID) AS produit_montant_marge_par_panier

            FROM
                v_sales_model
            GROUP BY
                HOTEL_CODE,
                TYPE,
                GAMME,
                NOM_PRODUIT
            ORDER BY 
                hotel_code,
                type,
                gamme,
                produit
            );
        """)


    def create_or_replace_view_v_produit_paniers_ventes_mois(self):
        # Cette fonction permet de calculer des indicateurs ratio par mois par rapport au nombre de mois d'activité

        # s'assurer que les vues impliquées dans le calcul existent
        self.create_if_not_exists_views(["v_produit_paniers_ventes", "v_nombre_mois"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme et produit par mois
        self.con.sql("""
            CREATE OR REPLACE VIEW v_produit_paniers_ventes_mois AS (
            SELECT 
                v_produit_paniers_ventes.*,
                nombre_mois,

                produit_nombre_ventes / nombre_mois AS produit_nombre_ventes_par_mois,
                produit_nombre_paniers / nombre_mois AS produit_nombre_paniers_par_mois,

                produit_montant_ventes / nombre_mois AS produit_montant_ventes_par_mois,
                produit_montant_achats / nombre_mois AS produit_montant_achats_par_mois,
                produit_montant_marge / nombre_mois AS produit_montant_marge_par_mois,
               
            FROM 
                v_produit_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_produit_paniers_ventes.hotel_code= v_nombre_mois.hotel_code
            );
        """)



    def create_or_replace_view_v_gamme_paniers_ventes(self):

        # s'assurer que la vue sur les données utilisées pour la modélisation existe
        self.create_if_not_exists_views(["v_sales_model"])
                        
        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme
        self.con.sql("""
            CREATE OR REPLACE VIEW v_gamme_paniers_ventes AS (
            SELECT 
                HOTEL_CODE AS hotel_code,
                TYPE as type,
                GAMME AS gamme,
                
                SUM(QUANTITE) AS gamme_nombre_ventes,
                COUNT(DISTINCT ORDER_ID) AS gamme_nombre_paniers,
                COUNT(DISTINCT NOM_PRODUIT) AS gamme_nombre_produits,
                
                SUM(PRIX_TTC) AS gamme_montant_ventes,
                SUM(PRIX_TTC_MARCHE) AS gamme_montant_achats,
                SUM(MARGE) AS gamme_montant_marge,

                SUM(PRIX_TTC) / SUM(QUANTITE) AS gamme_montant_par_vente,
                SUM(PRIX_TTC_MARCHE) / SUM(QUANTITE) AS gamme_montant_achats_par_vente,
                SUM(MARGE) / SUM(QUANTITE) AS gamme_montant_marge_par_vente,

                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS gamme_nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS gamme_montant_ventes_par_panier,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT ORDER_ID) AS gamme_montant_achats_par_panier,
                SUM(MARGE) / COUNT(DISTINCT ORDER_ID) AS gamme_montant_marge_par_panier,

                SUM(QUANTITE) / COUNT(DISTINCT NOM_PRODUIT) AS gamme_nombre_ventes_par_produit,
                SUM(PRIX_TTC) / COUNT(DISTINCT NOM_PRODUIT) AS gamme_montant_ventes_par_produit,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT NOM_PRODUIT) AS gamme_montant_achats_par_produit,
                SUM(MARGE) / COUNT(DISTINCT NOM_PRODUIT) AS gamme_montant_marge_par_produit

            FROM
                v_sales_model
            GROUP BY
                HOTEL_CODE,
                TYPE,
                GAMME
            ORDER BY 
                hotel_code,
                type,
                gamme
            );
        """)


    def create_or_replace_view_v_gamme_paniers_ventes_mois(self):
        """Cette fonction permet de calculer des indicateurs ratio par mois par rapport au nombre de mois d'activité"""

        # s'assurer que les vues impliquées dans le calcul existent
        self.create_if_not_exists_views(["v_gamme_paniers_ventes", "v_nombre_mois"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme par mois
        self.con.sql("""
            CREATE OR REPLACE VIEW v_gamme_paniers_ventes_mois AS (
            SELECT 
                v_gamme_paniers_ventes.*,
                nombre_mois,

                gamme_nombre_ventes / nombre_mois AS gamme_nombre_ventes_par_mois,
                gamme_nombre_paniers / nombre_mois AS gamme_nombre_paniers_par_mois,

                gamme_montant_ventes / nombre_mois AS gamme_montant_ventes_par_mois,
                gamme_montant_achats / nombre_mois AS gamme_montant_achats_par_mois,
                gamme_montant_marge / nombre_mois AS gamme_montant_marge_par_mois

            FROM 
                v_gamme_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_gamme_paniers_ventes.hotel_code= v_nombre_mois.hotel_code
            );
        """)
 

    def create_or_replace_view_v_type_paniers_ventes(self):

        # s'assurer que la vue sur les données utilisées pour la modélisation existe
        self.create_if_not_exists_views(["v_sales_model"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B)
        self.con.sql("""
            CREATE OR REPLACE VIEW v_type_paniers_ventes AS (
            SELECT 
                HOTEL_CODE AS hotel_code,
                TYPE AS type,

                SUM(QUANTITE) AS type_nombre_ventes,
                COUNT(DISTINCT ORDER_ID) AS type_nombre_paniers,
                COUNT(DISTINCT NOM_PRODUIT) AS type_nombre_produits,
                COUNT(DISTINCT GAMME) AS type_nombre_gammes,


                SUM(PRIX_TTC) AS type_montant_ventes,
                SUM(PRIX_TTC_MARCHE) AS type_montant_achats,
                SUM(MARGE) AS type_montant_marge,

                SUM(PRIX_TTC) / SUM(QUANTITE) AS type_montant_par_vente,
                SUM(PRIX_TTC_MARCHE) / SUM(QUANTITE) AS type_montant_achats_par_vente,
                SUM(MARGE) / SUM(QUANTITE) AS type_montant_marge_par_vente,

                
                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS type_nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS type_montant_ventes_par_panier,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT ORDER_ID) AS type_montant_achats_par_panier,
                SUM(MARGE) / COUNT(DISTINCT ORDER_ID) AS type_montant_marge_par_panier,


                SUM(QUANTITE) / COUNT(DISTINCT NOM_PRODUIT) AS type_nombre_ventes_par_produit,
                SUM(PRIX_TTC) / COUNT(DISTINCT NOM_PRODUIT) AS type_montant_ventes_par_produit,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT NOM_PRODUIT) AS type_montant_achats_par_produit,
                SUM(MARGE) / COUNT(DISTINCT NOM_PRODUIT) AS type_montant_marge_par_produit,


                SUM(QUANTITE) / COUNT(DISTINCT GAMME) AS type_nombre_ventes_par_gamme,
                SUM(PRIX_TTC) / COUNT(DISTINCT GAMME) AS type_montant_ventes_par_gamme,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT GAMME) AS type_montant_achats_par_gamme,
                SUM(MARGE) / COUNT(DISTINCT GAMME) AS type_montant_marge_par_gamme


            FROM
                v_sales_model
            GROUP BY
                HOTEL_CODE,
                TYPE
            ORDER BY 
                hotel_code,
                type
            );
        """)
    
    
    def create_or_replace_view_v_type_paniers_ventes_mois(self):
        """Cette fonction permet de calculer des indicateurs ratio par mois par rapport au nombre de mois d'activité"""

        # s'assurer que les vues impliquées dans le calcul existent
        self.create_if_not_exists_views(["v_type_paniers_ventes", "v_nombre_mois"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) par mois
        self.con.sql("""
            CREATE OR REPLACE VIEW v_type_paniers_ventes_mois AS (
            SELECT 
                v_type_paniers_ventes.*,
                nombre_mois,

                type_nombre_ventes / nombre_mois AS type_nombre_ventes_par_mois,
                type_nombre_paniers / nombre_mois AS type_nombre_paniers_par_mois,

                type_montant_ventes / nombre_mois AS type_montant_ventes_par_mois,
                type_montant_achats / nombre_mois AS type_montant_achats_par_mois,
                type_montant_marge / nombre_mois AS type_montant_marge_par_mois

            FROM 
                v_type_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_type_paniers_ventes.hotel_code= v_nombre_mois.hotel_code
            );
        """)
        
    
    
    def create_or_replace_view_v_paniers_ventes(self):

        # s'assurer que la vue sur les données utilisées pour la modélisation existe
        self.create_if_not_exists_views(["v_sales_model"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel
        self.con.sql("""
            CREATE OR REPLACE VIEW v_paniers_ventes AS (
            SELECT 
                HOTEL_CODE AS hotel_code,

                SUM(QUANTITE) AS nombre_ventes,
                COUNT(DISTINCT ORDER_ID) AS nombre_paniers,
                COUNT(DISTINCT NOM_PRODUIT) AS nombre_produits,
                COUNT(DISTINCT GAMME) AS nombre_gammes,
                COUNT(DISTINCT TYPE) AS nombre_types,

                

                SUM(PRIX_TTC) AS montant_ventes,
                SUM(PRIX_TTC_MARCHE) AS montant_achats,
                SUM(MARGE) AS montant_marge,

                SUM(PRIX_TTC) / SUM(QUANTITE) AS montant_par_vente,
                SUM(PRIX_TTC_MARCHE) / SUM(QUANTITE) AS montant_achat_par_vente,
                SUM(MARGE) / SUM(QUANTITE) AS montant_marge_par_vente,
                
                

                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS montant_ventes_par_panier,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT ORDER_ID) AS montant_achats_par_panier,
                SUM(MARGE) / COUNT(DISTINCT ORDER_ID) AS montant_marge_par_panier,


                SUM(QUANTITE) / COUNT(DISTINCT NOM_PRODUIT) AS nombre_ventes_par_produit,
                SUM(PRIX_TTC) / COUNT(DISTINCT NOM_PRODUIT) AS montant_ventes_par_produit,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT NOM_PRODUIT) AS montant_achats_par_produit,
                SUM(MARGE) / COUNT(DISTINCT NOM_PRODUIT) AS montant_marge_par_produit,


                SUM(QUANTITE) / COUNT(DISTINCT GAMME) AS nombre_ventes_par_gamme,
                SUM(PRIX_TTC) / COUNT(DISTINCT GAMME) AS montant_ventes_par_gamme,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT GAMME) AS montant_achats_par_gamme,
                SUM(MARGE) / COUNT(DISTINCT GAMME) AS montant_marge_par_gamme,

                
                SUM(QUANTITE) / COUNT(DISTINCT TYPE) AS nombre_ventes_par_type,
                SUM(PRIX_TTC) / COUNT(DISTINCT TYPE) AS montant_ventes_par_type,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT TYPE) AS montant_achats_par_type,
                SUM(MARGE) / COUNT(DISTINCT TYPE) AS montant_marge_par_type,


                METRES_LINEAIRES / COUNT(DISTINCT NOM_PRODUIT) AS nombre_metres_lineaires_par_produit,

                COUNT(DISTINCT NOM_PRODUIT) / METRES_LINEAIRES AS nombre_produits_par_metre_lineaire,
                SUM(QUANTITE) / METRES_LINEAIRES AS nombre_ventes_par_metre_lineaire,
                SUM(PRIX_TTC) / METRES_LINEAIRES AS montant_ventes_par_metre_lineaire,
                SUM(PRIX_TTC_MARCHE) / METRES_LINEAIRES AS montant_achats_par_metre_lineaire,
                SUM(MARGE) / METRES_LINEAIRES AS montant_marge_par_metre_lineaire,
            
                1 / COUNT(DISTINCT NOM_PRODUIT) AS produit_part_des_produits
            
            FROM
                v_sales_model
            GROUP BY
                HOTEL_CODE,
                METRES_LINEAIRES
            ORDER BY 
                hotel_code
            );
        """)


    def create_or_replace_view_v_paniers_ventes_mois(self):
        """Cette fonction permet de calculer des indicateurs ratio par mois par rapport au nombre de mois d'activité"""

        # s'assurer que les vues impliquées dans le calcul existent
        self.create_if_not_exists_views(["v_paniers_ventes", "v_nombre_mois"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel par mois
        self.con.sql("""
            CREATE OR REPLACE VIEW v_paniers_ventes_mois AS (
            SELECT 
                v_paniers_ventes.*,
                nombre_mois,

                nombre_ventes / nombre_mois AS nombre_ventes_par_mois,
                nombre_paniers / nombre_mois AS nombre_paniers_par_mois,

                montant_ventes / nombre_mois AS montant_ventes_par_mois,
                montant_achats / nombre_mois AS montant_achats_par_mois,
                montant_marge / nombre_mois AS montant_marge_par_mois

            FROM 
                v_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_paniers_ventes.hotel_code= v_nombre_mois.hotel_code
            );
        """)

            
    def create_or_replace_view_v_sales_model_base(self):
        """Cette fonction permet de réaliser la jointure de différentes vues créées afin de construire une vue de modélisation de base assez complète"""

        # s'assurer que les vues impliquées dans le calcul existent
        self.create_if_not_exists_views(["v_refs", "v_produit_paniers_ventes_mois", "v_gamme_paniers_ventes_mois", "v_type_paniers_ventes_mois", "v_paniers_ventes_mois"])
                    
        # vue template pour la modélisation
        self.con.sql("""
            CREATE OR REPLACE VIEW v_sales_model_base AS (
                SELECT 
                    *,
                    gamme_nombre_produits / nombre_produits AS gamme_part_des_produits,
                    type_nombre_produits / nombre_produits AS type_part_des_produits,

                    produit_nombre_ventes / nombre_ventes AS produit_part_nombre_ventes,
                    produit_montant_ventes / montant_ventes AS produit_part_montant_ventes,
                    produit_montant_achats / montant_achats AS produit_part_montant_achats,
                    produit_montant_marge / montant_marge AS produit_part_montant_marge,

                    gamme_nombre_ventes / nombre_ventes AS gamme_part_nombre_ventes,
                    gamme_montant_ventes / montant_ventes AS gamme_part_montant_ventes,
                    gamme_montant_achats / montant_achats AS gamme_part_montant_achats,
                    gamme_montant_marge / montant_marge AS gamme_part_montant_marge,

                    type_nombre_ventes / nombre_ventes AS type_part_nombre_ventes,
                    type_montant_ventes / montant_ventes AS type_part_montant_ventes,
                    type_montant_achats / montant_achats AS type_part_montant_achats,
                    type_montant_marge / montant_marge AS type_part_montant_marge,


                    produit_nombre_paniers / nombre_paniers AS produit_part_des_paniers,              
                    gamme_nombre_paniers / nombre_paniers AS gamme_part_des_paniers,
                    type_nombre_paniers / nombre_paniers AS type_part_des_paniers,



                FROM 
                    v_refs

                INNER JOIN
                    v_produit_paniers_ventes_mois
                USING 
                    (hotel_code)
                
                INNER JOIN 
                    v_gamme_paniers_ventes_mois
                USING 
                    (hotel_code, nombre_mois, type, gamme)
                
                INNER JOIN 
                    v_type_paniers_ventes_mois
                USING 
                    (hotel_code, nombre_mois, type)
                
                INNER JOIN 
                    v_paniers_ventes_mois
                USING 
                    (hotel_code, nombre_mois)
            );
        """)

    @classmethod
    def main(cls):
        """Cette fonction représente la fonction principale qui exploite cette classe et qu'elle faut exécuter"""

        base = SalesPilBase() # on instancie un objet de la classe qui gère les données pilote de base

        print(base.view_df("v_sales").shape) # on inspecte le shape de la dataframe des ventes brutes
        display(base.view_df("v_sales").head(3)) # on inspecte un echantillons de données de cette dataframe

        print(base.view_df("v_sales_model").shape) # on inspecte le shape de la dataframe des ventes brutes des donnéees de modélisation
        display(base.view_df("v_sales_model").head(3)) # on inspecte un echantillons de données de cette dataframe
        

        print(base.view_df("v_sales_model_base").shape) # on inspecte le shape de la dataframe de modélisaiton de base 
        display(base.view_df("v_sales_model_base").head(3)) # on inspecte un echantillons de données de cette dataframe

        base.create_if_not_exists_table_from_view("t_sales_model", "v_sales_model") # on sauvegarde physiquement la table source des données de modélisation
        base.create_if_not_exists_table_from_view("t_sales_model_base", "v_sales_model_base") # on sauvegarde physiquement la table des indicateurs de modélisation

        base.close_con() # on cloture correctement la connexion pour éviter tout conflit


In [2]:
SalesPilBase.main() # exécuter la fonction principale de la classe permettant de construire la base de données de modélisation

(130566, 28)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Tongs Femme 100 Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Casquette Enfant -Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque Easybreath de Surface Adulte - 500 Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


(109342, 28)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Tongs Femme 100 Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Casquette Enfant -Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque Easybreath de Surface Adulte - 500 Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


(872, 127)


,hotel_code,hotel_name,solution,metres_lineaires,type,gamme,produit,produit_nombre_ventes,produit_nombre_paniers,produit_montant_ventes,produit_montant_achats,produit_montant_marge,produit_montant_par_vente,produit_montant_achats_par_vente,produit_montant_marge_par_vente,produit_nombre_ventes_par_panier,produit_montant_ventes_par_panier,produit_montant_achats_par_panier,produit_montant_marge_par_panier,nombre_mois,produit_nombre_ventes_par_mois,produit_nombre_paniers_par_mois,produit_montant_ventes_par_mois,produit_montant_achats_par_mois,produit_montant_marge_par_mois,gamme_nombre_ventes,gamme_nombre_paniers,gamme_nombre_produits,gamme_montant_ventes,gamme_montant_achats,gamme_montant_marge,gamme_montant_par_vente,gamme_montant_achats_par_vente,gamme_montant_marge_par_vente,gamme_nombre_ventes_par_panier,gamme_montant_ventes_par_panier,gamme_montant_achats_par_panier,gamme_montant_marge_par_panier,gamme_nombre_ventes_par_produit,gamme_montant_ventes_par_produit,gamme_montant_achats_par_produit,gamme_montant_marge_par_produit,gamme_nombre_ventes_par_mois,gamme_nombre_paniers_par_mois,gamme_montant_ventes_par_mois,gamme_montant_achats_par_mois,gamme_montant_marge_par_mois,type_nombre_ventes,type_nombre_paniers,type_nombre_produits,type_nombre_gammes,type_montant_ventes,type_montant_achats,type_montant_marge,type_montant_par_vente,type_montant_achats_par_vente,type_montant_marge_par_vente,type_nombre_ventes_par_panier,type_montant_ventes_par_panier,type_montant_achats_par_panier,type_montant_marge_par_panier,type_nombre_ventes_par_produit,type_montant_ventes_par_produit,type_montant_achats_par_produit,type_montant_marge_par_produit,type_nombre_ventes_par_gamme,type_montant_ventes_par_gamme,type_montant_achats_par_gamme,type_montant_marge_par_gamme,type_nombre_ventes_par_mois,type_nombre_paniers_par_mois,type_montant_ventes_par_mois,type_montant_achats_par_mois,type_montant_marge_par_mois,nombre_ventes,nombre_paniers,nombre_produits,nombre_gammes,nombre_types,montant_ventes,montant_achats,montant_marge,montant_par_vente,montant_achat_par_vente,montant_marge_par_vente,nombre_ventes_par_panier,montant_ventes_par_panier,montant_achats_par_panier,montant_marge_par_panier,nombre_ventes_par_produit,montant_ventes_par_produit,montant_achats_par_produit,montant_marge_par_produit,nombre_ventes_par_gamme,montant_ventes_par_gamme,montant_achats_par_gamme,montant_marge_par_gamme,nombre_ventes_par_type,montant_ventes_par_type,montant_achats_par_type,montant_marge_par_type,nombre_metres_lineaires_par_produit,nombre_produits_par_metre_lineaire,nombre_ventes_par_metre_lineaire,montant_ventes_par_metre_lineaire,montant_achats_par_metre_lineaire,montant_marge_par_metre_lineaire,produit_part_des_produits,nombre_ventes_par_mois,nombre_paniers_par_mois,montant_ventes_par_mois,montant_achats_par_mois,montant_marge_par_mois,gamme_part_des_produits,type_part_des_produits,produit_part_nombre_ventes,produit_part_montant_ventes,produit_part_montant_achats,produit_part_montant_marge,gamme_part_nombre_ventes,gamme_part_montant_ventes,gamme_part_montant_achats,gamme_part_montant_marge,type_part_nombre_ventes,type_part_montant_ventes,type_part_montant_achats,type_part_montant_marge
0,H0373,Mercure Paris Montmartre Sacré-Cœur,connected,6.0,NON-F&B,SOUVENIRS,Tote-Bag Parisienne,6.0,5,192.0,192.0,0.0,32.0,32.00,0.00,1.2,38.4,38.4,0.0,24,0.250000,0.208333,8.00000,8.00000,0.000,13.0,12,3,329.0,329.0,-2.131628e-14,25.307692,25.307692,-1.639714e-15,1.083333,27.416667,27.416667,-1.776357e-15,4.333333,109.666667,109.666667,-7.105427e-15,0.541667,0.500000,13.708333,13.708333,-8.881784e-16,218.0,175,48,5,3435.4402,3495.10225,-59.66205,15.758900,16.032579,-0.273679,1.245714,19.631087,19.972013,-0.340926,4.541667,71.571671,72.814630,-1.242959,43.60,687.08804,699.02045,-11.93241,9.083333,7.291667,143.143342,145.629260,-2.485919,15759.0,8278,146,9,2,84373.3057,83953.10225,420.20345,5.353976,5.327312,0.026664,1.903721,10.192475,10.141713,0.050761,107.938356,577.899354,575.021

In [39]:
class SalesPilSim(SalesPilBase):
    """
    Cette classe permet d'exécuter des simulations en relançant les mêmes calculs de la classe mère sur des vues de données modifiées en fonction de ce qu'on souhaite simuler.
    L'objectif de cette classe est de construire plusieurs jeux de données de simulation qui vont être exploitées par la suite par une modèle de machine learning.
    """
    def __init__(self, db_con_str : str = "duckdb/pilotes/sim/sim.duckdb",  sales_file_path : str = "data/hotel_sales_raw_extended_data.xlsx"):
        super().__init__(db_con_str, sales_file_path)


    def remove_elements(self, champs : str, elements : list[str]):
        """Cette fonction permet de calculer les indicateurs de simulation si certains éléments (produits, gammes, types, etc) des données de ventes brutes n'existaient pas"""

        # convertir la liste des éléments à retirer en une chaine de caractères pour l'inclure dans la requête sql
        list_str = ",".join([f"'{elm}'" for elm in elements])

        # s'assurer que la table des données de modélisation existe déjà
        self.create_if_not_exists_view("v_sales_model") # d'abord la vue
        self.create_if_not_exists_table_from_view("t_sales_model", "v_sales_model") # puis la table à partir de la vue

        # s'assurer que la table figée des indicateurs avant toute simulation existe déjà
        self.create_if_not_exists_view("v_sales_model_base") # d'abord la vue
        self.create_if_not_exists_table_from_view("t_sales_model_base", "v_sales_model_base") # puis la table à partir de la vue

        # supprimer les vues afin de les recréer en partant de la vue source après avoir retiré les éléments
        self.drop_if_exists_views(self.views_df()["table_name"])

        # calculer la nouvelle quantité de demandes disponible correspondant aux clients acheteurs qui n'achetéront plus les produits retirés 
        self.con.sql(f"""
            CREATE OR REPLACE VIEW v_demande_quantite AS (
                SELECT
                    HOTEL_CODE,
                    COALESCE(SUM(CASE
                        WHEN {champs} IN ({list_str}) THEN QUANTITE
                        ELSE 0
                    END), 0) AS demande_quantite
                FROM
                    t_sales_model
                GROUP BY
                    hotel_code
                ORDER BY
                    hotel_code
            )
        """)

        # récupérer la part dans le nombre de ventes pour chaque produit vendu par chaque hotel et l'associer à la nouvelle quantité de demande
        sim.con.sql("""
            CREATE OR REPLACE VIEW v_part_demande_quantite AS (
                SELECT
                    HOTEL_CODE,
                    produit AS NOM_PRODUIT,
                    produit_nombre_ventes,
                    produit_part_nombre_ventes,
                    demande_quantite,
                    demande_quantite * produit_part_nombre_ventes AS part_demande_quantite
                FROM
                    v_demande_quantite
                LEFT JOIN
                    t_sales_model_base
                USING
                    (HOTEL_CODE)
                ORDER BY
                    HOTEL_CODE,
                    NOM_PRODUIT
            )
        """)

        # créer une liste de ventes fictives correspondant à la demande à intégrer et respectant le comportement des clients modélisés
        sim.con.sql("""
            CREATE OR REPLACE VIEW v_sales_model_demande AS (
                SELECT 
                    SOLUTION,
                    HOTEL_CODE,
                    HOTEL_NAME,
                    METRES_LINEAIRES,
                    NOM_BOUTIQUE, 
                    TYPE,
                    TYPE_RAW,
                    GAMME,
                    GAMME_RAW,
                    NOM_PRODUIT,
                    NOM_PRODUIT_RAW,
                    CATEGORIE,
                    OPERATEUR,
                    MACHINE,
                    DATE,
                    HEURE,
                    STATUT,
                    CODE_EAN,
                    (QUANTITE / produit_nombre_ventes) * part_demande_quantite AS QUANTITE,
                    (PRIX_HT / produit_nombre_ventes)  * part_demande_quantite AS PRIX_HT,
                    VAT,
                    (PRIX_TTC / produit_nombre_ventes)  * part_demande_quantite AS PRIX_TTC,
                    MARQUE,
                    FOURNISSEUR,
                    - ORDER_ID AS ORDER_ID,
                    TEMPERATURE,
                    (PRIX_TTC_MARCHE / produit_nombre_ventes)  * part_demande_quantite AS PRIX_TTC_MARCHE,
                    (MARGE / produit_nombre_ventes)  * part_demande_quantite AS MARGE
                FROM 
                    t_sales_model
                RIGHT JOIN
                (
                    SELECT
                        *
                    FROM
                        v_part_demande_quantite
                    WHERE
                        part_demande_quantite > 0
                )
                USING
                    (HOTEL_CODE, NOM_PRODUIT)
            )
            """)

        # créer la vue des données qui sera utilisée pour faire le calcul des indicateurs sur les données de modélisation
        self.con.sql(f"""
            CREATE OR REPLACE VIEW v_sales_model AS (
                SELECT
                    *
                FROM
                    t_sales_model
                WHERE
                    {champs} NOT IN ({list_str})

                UNION ALL

                SELECT
                    *
                FROM
                    v_sales_model_demande
                WHERE
                    {champs} NOT IN ({list_str})
            )
        """)

        # recréer la vue des indicateurs et toutes les vues intermédiaires en partant de la vue source modifiée
        self.create_if_not_exists_view("v_sales_model_base")

        

        


    def remove_produits(self, produits : list[str]):
        """Cette fonction permet de créer les tables qui simule le retrait d'une liste de produits"""
        self.remove_elements("NOM_PRODUIT", produits)
        

    def remove_gammes(self, gammes : list[str]):
        """Cette fonction permet de créer les tables qui simule le retrait d'une liste de gammes"""
        self.remove_elements("GAMME", gammes)


    def remove_types(self, types : list[str]):
        """
        Cette fonction permet de créer les tables qui simule le retrait d'une liste de types.
        Pour le moment il n y a que deux type 'F&B' et 'Non F&B', donc la liste ne peut contenir qu'un seul élément sinon ça n'aura pas d'intérêt. 
        On garde la fonction dans ce format pour rester alligné avec les autres fonctions.
        """
        self.remove_elements("TYPE", types)


    @classmethod
    def main(cls):
        """Cette fonction représente la fonction principale qui exploite cette classe et qu'elle faut exécuter"""

        # on s'assure qu'on a d'abord les données de modélisation de base avant de commencer la simulation
        super().main()

        sim = SalesPilSim() # instancier un objet de la classe de simulation

        
        # sim.create_if_not_exists_view("v_produits") # s'assurer que la vue sur les produits existe
        # produits = sim.con.sql("SELECT DISTINCT produit from v_produits").df()["produit"] # récupérer les produits distincts indépendamment de l'hotel
        
        # for produit in produits: # parcourir la liste des produits pour faire une simulation un par un 
        #     sim = SalesPilSim() # repartir d'une nouvelle instance avec les vues de base 
        #     sim.remove_produits([produit]) # simuler la suppression 
        #     break

In [40]:
SalesPilSim().main()

(130566, 28)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Tongs Femme 100 Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Casquette Enfant -Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque Easybreath de Surface Adulte - 500 Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


(109342, 28)


,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Tongs Femme 100 Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Casquette Enfant -Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque Easybreath de Surface Adulte - 500 Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


(872, 127)


,hotel_code,hotel_name,solution,metres_lineaires,type,gamme,produit,produit_nombre_ventes,produit_nombre_paniers,produit_montant_ventes,produit_montant_achats,produit_montant_marge,produit_montant_par_vente,produit_montant_achats_par_vente,produit_montant_marge_par_vente,produit_nombre_ventes_par_panier,produit_montant_ventes_par_panier,produit_montant_achats_par_panier,produit_montant_marge_par_panier,nombre_mois,produit_nombre_ventes_par_mois,produit_nombre_paniers_par_mois,produit_montant_ventes_par_mois,produit_montant_achats_par_mois,produit_montant_marge_par_mois,gamme_nombre_ventes,gamme_nombre_paniers,gamme_nombre_produits,gamme_montant_ventes,gamme_montant_achats,gamme_montant_marge,gamme_montant_par_vente,gamme_montant_achats_par_vente,gamme_montant_marge_par_vente,gamme_nombre_ventes_par_panier,gamme_montant_ventes_par_panier,gamme_montant_achats_par_panier,gamme_montant_marge_par_panier,gamme_nombre_ventes_par_produit,gamme_montant_ventes_par_produit,gamme_montant_achats_par_produit,gamme_montant_marge_par_produit,gamme_nombre_ventes_par_mois,gamme_nombre_paniers_par_mois,gamme_montant_ventes_par_mois,gamme_montant_achats_par_mois,gamme_montant_marge_par_mois,type_nombre_ventes,type_nombre_paniers,type_nombre_produits,type_nombre_gammes,type_montant_ventes,type_montant_achats,type_montant_marge,type_montant_par_vente,type_montant_achats_par_vente,type_montant_marge_par_vente,type_nombre_ventes_par_panier,type_montant_ventes_par_panier,type_montant_achats_par_panier,type_montant_marge_par_panier,type_nombre_ventes_par_produit,type_montant_ventes_par_produit,type_montant_achats_par_produit,type_montant_marge_par_produit,type_nombre_ventes_par_gamme,type_montant_ventes_par_gamme,type_montant_achats_par_gamme,type_montant_marge_par_gamme,type_nombre_ventes_par_mois,type_nombre_paniers_par_mois,type_montant_ventes_par_mois,type_montant_achats_par_mois,type_montant_marge_par_mois,nombre_ventes,nombre_paniers,nombre_produits,nombre_gammes,nombre_types,montant_ventes,montant_achats,montant_marge,montant_par_vente,montant_achat_par_vente,montant_marge_par_vente,nombre_ventes_par_panier,montant_ventes_par_panier,montant_achats_par_panier,montant_marge_par_panier,nombre_ventes_par_produit,montant_ventes_par_produit,montant_achats_par_produit,montant_marge_par_produit,nombre_ventes_par_gamme,montant_ventes_par_gamme,montant_achats_par_gamme,montant_marge_par_gamme,nombre_ventes_par_type,montant_ventes_par_type,montant_achats_par_type,montant_marge_par_type,nombre_metres_lineaires_par_produit,nombre_produits_par_metre_lineaire,nombre_ventes_par_metre_lineaire,montant_ventes_par_metre_lineaire,montant_achats_par_metre_lineaire,montant_marge_par_metre_lineaire,produit_part_des_produits,nombre_ventes_par_mois,nombre_paniers_par_mois,montant_ventes_par_mois,montant_achats_par_mois,montant_marge_par_mois,gamme_part_des_produits,type_part_des_produits,produit_part_nombre_ventes,produit_part_montant_ventes,produit_part_montant_achats,produit_part_montant_marge,gamme_part_nombre_ventes,gamme_part_montant_ventes,gamme_part_montant_achats,gamme_part_montant_marge,type_part_nombre_ventes,type_part_montant_ventes,type_part_montant_achats,type_part_montant_marge
0,H0373,Mercure Paris Montmartre Sacré-Cœur,connected,6.0,NON-F&B,SOUVENIRS,Tote-Bag Parisienne,6.0,5,192.0,192.0,0.0,32.0,32.00,0.00,1.2,38.4,38.4,0.0,24,0.250000,0.208333,8.00000,8.00000,0.000,13.0,12,3,329.0,329.0,-2.131628e-14,25.307692,25.307692,-1.639714e-15,1.083333,27.416667,27.416667,-1.776357e-15,4.333333,109.666667,109.666667,-7.105427e-15,0.541667,0.500000,13.708333,13.708333,-8.881784e-16,218.0,175,48,5,3435.4402,3495.10225,-59.66205,15.758900,16.032579,-0.273679,1.245714,19.631087,19.972013,-0.340926,4.541667,71.571671,72.814630,-1.242959,43.60,687.08804,699.02045,-11.93241,9.083333,7.291667,143.143342,145.629260,-2.485919,15759.0,8278,146,9,2,84373.3057,83953.10225,420.20345,5.353976,5.327312,0.026664,1.903721,10.192475,10.141713,0.050761,107.938356,577.899354,575.021

In [41]:
sim = SalesPilSim() # instancier un objet de la classe de simulation
sim.view_df("v_sales_model_base").head(3)

,hotel_code,hotel_name,solution,metres_lineaires,type,gamme,produit,produit_nombre_ventes,produit_nombre_paniers,produit_montant_ventes,produit_montant_achats,produit_montant_marge,produit_montant_par_vente,produit_montant_achats_par_vente,produit_montant_marge_par_vente,produit_nombre_ventes_par_panier,produit_montant_ventes_par_panier,produit_montant_achats_par_panier,produit_montant_marge_par_panier,nombre_mois,produit_nombre_ventes_par_mois,produit_nombre_paniers_par_mois,produit_montant_ventes_par_mois,produit_montant_achats_par_mois,produit_montant_marge_par_mois,gamme_nombre_ventes,gamme_nombre_paniers,gamme_nombre_produits,gamme_montant_ventes,gamme_montant_achats,gamme_montant_marge,gamme_montant_par_vente,gamme_montant_achats_par_vente,gamme_montant_marge_par_vente,gamme_nombre_ventes_par_panier,gamme_montant_ventes_par_panier,gamme_montant_achats_par_panier,gamme_montant_marge_par_panier,gamme_nombre_ventes_par_produit,gamme_montant_ventes_par_produit,gamme_montant_achats_par_produit,gamme_montant_marge_par_produit,gamme_nombre_ventes_par_mois,gamme_nombre_paniers_par_mois,gamme_montant_ventes_par_mois,gamme_montant_achats_par_mois,gamme_montant_marge_par_mois,type_nombre_ventes,type_nombre_paniers,type_nombre_produits,type_nombre_gammes,type_montant_ventes,type_montant_achats,type_montant_marge,type_montant_par_vente,type_montant_achats_par_vente,type_montant_marge_par_vente,type_nombre_ventes_par_panier,type_montant_ventes_par_panier,type_montant_achats_par_panier,type_montant_marge_par_panier,type_nombre_ventes_par_produit,type_montant_ventes_par_produit,type_montant_achats_par_produit,type_montant_marge_par_produit,type_nombre_ventes_par_gamme,type_montant_ventes_par_gamme,type_montant_achats_par_gamme,type_montant_marge_par_gamme,type_nombre_ventes_par_mois,type_nombre_paniers_par_mois,type_montant_ventes_par_mois,type_montant_achats_par_mois,type_montant_marge_par_mois,nombre_ventes,nombre_paniers,nombre_produits,nombre_gammes,nombre_types,montant_ventes,montant_achats,montant_marge,montant_par_vente,montant_achat_par_vente,montant_marge_par_vente,nombre_ventes_par_panier,montant_ventes_par_panier,montant_achats_par_panier,montant_marge_par_panier,nombre_ventes_par_produit,montant_ventes_par_produit,montant_achats_par_produit,montant_marge_par_produit,nombre_ventes_par_gamme,montant_ventes_par_gamme,montant_achats_par_gamme,montant_marge_par_gamme,nombre_ventes_par_type,montant_ventes_par_type,montant_achats_par_type,montant_marge_par_type,nombre_metres_lineaires_par_produit,nombre_produits_par_metre_lineaire,nombre_ventes_par_metre_lineaire,montant_ventes_par_metre_lineaire,montant_achats_par_metre_lineaire,montant_marge_par_metre_lineaire,produit_part_des_produits,nombre_ventes_par_mois,nombre_paniers_par_mois,montant_ventes_par_mois,montant_achats_par_mois,montant_marge_par_mois,gamme_part_des_produits,type_part_des_produits,produit_part_nombre_ventes,produit_part_montant_ventes,produit_part_montant_achats,produit_part_montant_marge,gamme_part_nombre_ventes,gamme_part_montant_ventes,gamme_part_montant_achats,gamme_part_montant_marge,type_part_nombre_ventes,type_part_montant_ventes,type_part_montant_achats,type_part_montant_marge,produit_part_des_paniers,gamme_part_des_paniers,type_part_des_paniers
0,H0373,Mercure Paris Montmartre Sacré-Cœur,connected,6.0,F&B,SUGARY FOOD,Set de Thé (3 x 25g),4.000000,3,48.000000,48.000000,0.000000,12.000000,12.000000,0.000000,1.333333,16.000000,16.000000,0.000000,24,0.166667,0.125000,2.000000,2.000000,0.000000,2230.000000,1508,24,5532.169500,5544.000000,-11.830500,2.480793,2.486099,-0.005305,1.478780,3.668547,3.676393,-0.007845,92.916667,230.507062,231.000000,-0.492938,92.916667,62.833333,230.507062,231.000000,-0.492938,15541.000000,8114,98,5,80937.865500,80458.000000,479.865500,5.208022,5.177144,0.030877,1.915332,9.975088,9.915948,0.059140,158.581633,825.896587,821.000000,4.896587,3108.200000,16187.573100,16091.600000,95.973100,647.541667,338.083333,3372.411062,3352.416667,

In [42]:
produits = ["Tongs Femme 100 Noir"]
champs = "NOM_PRODUIT"
list_str = ",".join([f"'{produit}'" for produit in produits])
sim.remove_produits(produits)

In [43]:
sim.view_df("v_demande_quantite")

,HOTEL_CODE,demande_quantite
0,H0373,0.0
1,H2075,63.0
2,H3546,0.0
3,H5586,0.0
4,H6188,0.0
5,HB5I0,0.0
6,HB6A3,0.0


In [44]:
# créer la vue des données qui sera utilisée pour faire le calcul des indicateurs sur les données de modélisation
sim.con.sql(f"""
(SELECT
    *
FROM
    v_sales_model_base
WHERE
    HOTEL_CODE = 'H2075'
ORDER BY
    hotel_code,
    produit
LIMIT 
    3
)
UNION ALL
(
SELECT
    *
FROM
    t_sales_model_base
WHERE
    HOTEL_CODE = 'H2075'
ORDER BY
    hotel_code,
    produit
LIMIT
    3
)
""")

BinderException: Binder Error: Set operations can only apply to expressions with the same number of result columns

In [45]:
sim.view_df("v_part_demande_quantite")
sim.con.sql("SELECT * FROM v_part_demande_quantite WHERE demande_quantite >0")

┌────────────┬──────────────────────────────────────────────────────────────────────────────┬───────────────────────┬────────────────────────────┬──────────────────┬───────────────────────┐
│ HOTEL_CODE │                                 NOM_PRODUIT                                  │ produit_nombre_ventes │ produit_part_nombre_ventes │ demande_quantite │ part_demande_quantite │
│  varchar   │                                   varchar                                    │        int128         │           double           │      int128      │        double         │
├────────────┼──────────────────────────────────────────────────────────────────────────────┼───────────────────────┼────────────────────────────┼──────────────────┼───────────────────────┤
│ H2075      │ Balle de Massage et D'Acupression - TU                                       │                     2 │      0.0003952569169960474 │               63 │  0.024901185770750987 │
│ H2075      │ Balle de Massage et D'Acupression /

In [46]:
sim.view_df("v_sales_model").shape

(114276, 28)

In [47]:
sim.con.sql("SELECT * FROM t_sales_model").df().shape

(109342, 28)

In [57]:
sim.con.sql("""
SELECT
    HOTEL_CODE,
    ORDER_ID,
    SUM(QUANTITE) AS nombre_ventes_par_panier,
    COUNT(DISTINCT NOM_PRODUIT) AS nombre_produits_par_panier,
    COUNT(DISTINCT GAMME) AS nombre_gammes_par_panier,
    COUNT(DISTINCT TYPE) AS nombre_types_par_panier,

    SUM(PRIX_TTC) AS montant_ventes_par_panier,
    SUM(PRIX_TTC_MARCHE) AS montant_achats_par_panier,
    SUM(MARGE) AS montant_marge_par_panier
FROM
    t_sales_model
GROUP BY
    HOTEL_CODE,
    ORDER_ID
""")

# sim.con.sql("""
# SELECT
#     *
# FROM
#     t_sales_model

# WHERE
#     ORDER_ID = 49471

# """)




┌────────────┬──────────┬──────────────────────────┬────────────────────────────┬──────────────────────────┬─────────────────────────┬───────────────────────────┬───────────────────────────┬──────────────────────────┐
│ HOTEL_CODE │ ORDER_ID │ nombre_ventes_par_panier │ nombre_produits_par_panier │ nombre_gammes_par_panier │ nombre_types_par_panier │ montant_ventes_par_panier │ montant_achats_par_panier │ montant_marge_par_panier │
│  varchar   │  int64   │          int128          │           int64            │          int64           │          int64          │          double           │          double           │          double          │
├────────────┼──────────┼──────────────────────────┼────────────────────────────┼──────────────────────────┼─────────────────────────┼───────────────────────────┼───────────────────────────┼──────────────────────────┤
│ H2075      │     9281 │                        1 │                          1 │                        1 │                    

In [49]:
sim.view_df("v_part_demande_quantite")

,HOTEL_CODE,NOM_PRODUIT,produit_nombre_ventes,produit_part_nombre_ventes,demande_quantite,part_demande_quantite
0,H0373,1 Kit Couverts Bleu Nuit (binikit),2.0,0.000127,0.0,0.0
1,H0373,100g Chips Truffe Mpg,30.0,0.001904,0.0,0.0
2,H0373,100g Choc Noir Mpg M. Havelaar,8.0,0.000508,0.0,0.0
3,H0373,6 Barres Chocolat 125g,9.0,0.000571,0.0,0.0
4,H0373,6 Barres de Céréales au Chocolat 125g,4.0,0.000254,0.0,0.0
...,...,...,...,...,...,...
867,HB6A3,Special K Chocolat Noir,24.0,0.003480,0.0,0.0
868,HB6A3,Sprite,165.0,0.023927,0.0,0.0
869,HB6A3,Tao Pure Infusion Brun,4.0,0.000580,0.0,0.0
870,HB6A3,Tao Pure Infusion Rouge,1.0,0.000145,0.0,0.0


In [47]:
sim.con.sql("""
CREATE OR REPLACE VIEW v_sales_model_demande AS (
SELECT 
    SOLUTION,
    HOTEL_CODE,
    HOTEL_NAME,
    METRES_LINEAIRES,
    NOM_BOUTIQUE, 
    TYPE,
    TYPE_RAW,
    GAMME,
    GAMME_RAW,
    NOM_PRODUIT,
    NOM_PRODUIT_RAW,
    CATEGORIE,
    OPERATEUR,
    MACHINE,
    DATE,
    HEURE,
    STATUT,
    CODE_EAN,
    (QUANTITE / produit_nombre_ventes) * part_demande_quantite AS QUANTITE,
    (PRIX_HT / produit_nombre_ventes)  * part_demande_quantite AS PRIX_HT,
    VAT,
    (PRIX_TTC / produit_nombre_ventes)  * part_demande_quantite AS PRIX_TTC,
    MARQUE,
    FOURNISSEUR,
    - ORDER_ID AS ORDER_ID,
    TEMPERATURE,
    PRIX_TTC_MARCHE,
    (MARGE / produit_nombre_ventes)  * part_demande_quantite AS MARGE
FROM 
    t_sales_model
LEFT JOIN
    v_part_demande_quantite
USING
    (HOTEL_CODE, NOM_PRODUIT)
)
""")

In [13]:
sim.close_con()